# 🧹 Data Cleaning Notebook

---

## Objective
Clean and preprocess the World Real-Time Food Prices dataset for further analysis.

## Why Data Cleaning Matters
Data cleaning is **critical** in data analytics because:
- **Garbage In, Garbage Out**: Poor quality data leads to unreliable insights
- **Statistical Validity**: Clean data ensures statistical tests produce accurate results  
- **Machine Learning Performance**: ML models are sensitive to missing values, outliers, and inconsistent data
- **Reproducibility**: Well-documented cleaning steps allow others to verify and reproduce our analysis

## About the Dataset

| Item | Details |
|------|---------|
| **Source** | World Bank - Real-Time Food Prices |
| **Time period** | January 2007 to October 2023 |
| **Input file** | `data/raw/WLD_RTFP_country_2023-10-02.csv` |
| **Output file** | `data/cleaned/food_prices_cleaned.csv` |

## Steps
1. Load the raw data
2. Explore the data structure
3. Check for missing values and duplicates
4. Clean and transform data
5. Create new features (feature engineering)
6. Save cleaned data

---
## Step 1: Import Libraries

### Libraries Used:
- **pandas**: Industry-standard library for data manipulation and analysis. Provides DataFrame structure for tabular data.
- **numpy**: Fundamental package for numerical computing. Handles arrays and mathematical operations efficiently.
- **os**: Allows interaction with the operating system for file path management.

In [3]:
import pandas as pd
import numpy as np
import os

pd.set_option('display.max_columns', None)

print("✅ Libraries loaded successfully!")

✅ Libraries loaded successfully!


---
## Step 2: Load Raw Data

In [4]:
df = pd.read_csv('../data/raw/WLD_RTFP_country_2023-10-02.csv')

print("✅ Data loaded successfully!")
print(f"📊 Dataset size: {df.shape[0]:,} rows, {df.shape[1]} columns")

✅ Data loaded successfully!
📊 Dataset size: 4,798 rows, 8 columns


---
## Step 3: Explore the Data

### Why Exploratory Data Analysis (EDA)?
Before cleaning, we need to understand:
- **Data types**: Are columns numeric, categorical, or dates?
- **Shape**: How many records and features do we have?
- **Sample records**: What does the actual data look like?

This initial exploration guides our cleaning decisions.

In [5]:
print("👀 First 5 rows:")
df.head()

👀 First 5 rows:


,Open,High,Low,Close,Inflation,country,ISO3,date
0,0.53,0.54,0.53,0.53,NaN,Afghanistan,AFG,2007-01-01
1,0.53,0.54,0.53,0.53,NaN,Afghanistan,AFG,2007-02-01
2,0.54,0.54,0.53,0.53,NaN,Afghanistan,AFG,2007-03-01
3,0.53,0.55,0.53,0.55,NaN,Afghanistan,AFG,2007-04-01
4,0.56,0.57,0.56,0.57,NaN,Afghanistan,AFG,2007-05-01


In [6]:
print("📋 Column information:")
df.info()

📋 Column information:
<class 'pandas.DataFrame'>
RangeIndex: 4798 entries, 0 to 4797
Data columns (total 8 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Open       4734 non-null   float64
 1   High       4734 non-null   float64
 2   Low        4734 non-null   float64
 3   Close      4734 non-null   float64
 4   Inflation  4434 non-null   float64
 5   country    4798 non-null   str    
 6   ISO3       4798 non-null   str    
 7   date       4798 non-null   str    
dtypes: float64(5), str(3)
memory usage: 300.0 KB


In [7]:
print("🔢 Basic statistics:")
df.describe()

🔢 Basic statistics:


,Open,High,Low,Close,Inflation
count,4734.000000,4734.000000,4734.000000,4734.000000,4434.000000
mean,1.491880,1.536158,1.451056,1.492398,14.692346
std,4.652457,4.883312,4.439229,4.633321,35.910342
min,0.010000,0.010000,0.010000,0.010000,-31.470000
25%,0.740000,0.750000,0.720000,0.740000,-0.487500
50%,0.960000,0.980000,0.950000,0.960000,5.360000
75%,1.100000,1.120000,1.077500,1.100000,16.372500
max,102.460000,106.480000,94.420000,94.420000,363.100000


In [8]:
print(f"🌍 Countries in dataset: {df['country'].nunique()}")
print(sorted(df['country'].unique()))

🌍 Countries in dataset: 25
['Afghanistan', 'Burkina Faso', 'Burundi', 'Cameroon', 'Central African Republic', 'Chad', 'Congo, Dem. Rep.', 'Congo, Rep.', 'Gambia, The', 'Guinea-Bissau', 'Haiti', 'Iraq', 'Lao PDR', 'Lebanon', 'Liberia', 'Mali', 'Mozambique', 'Myanmar', 'Niger', 'Nigeria', 'Somalia', 'South Sudan', 'Sudan', 'Syrian Arab Republic', 'Yemen, Rep.']


---
## Step 4: Check for Missing Values & Duplicates

### Why Check for Missing Values?
- **Statistical Impact**: Missing values can bias calculations (e.g., mean, standard deviation)
- **ML Model Requirements**: Many algorithms cannot handle NaN values
- **Decision Point**: We must decide whether to impute, drop, or flag missing data

### Why Check for Duplicates?
- **Data Integrity**: Duplicates artificially inflate record counts and statistics
- **Fair Analysis**: Each observation should be counted only once

### Our Approach
For inflation column with ~7.6% missing values, we retain them as the other columns are complete. Inflation is calculated year-over-year, so the first 12 months naturally have NaN values.

In [9]:
print("🔍 Missing Values Report")
print("="*50)
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df) * 100).round(2)
missing_report = pd.DataFrame({'Missing': missing, 'Percentage': missing_pct})
print(missing_report)
print(f"\nTotal missing: {missing.sum():,}")

🔍 Missing Values Report
           Missing  Percentage
Open            64        1.33
High            64        1.33
Low             64        1.33
Close           64        1.33
Inflation      364        7.59
country          0        0.00
ISO3             0        0.00
date             0        0.00

Total missing: 620


In [10]:
print("🔄 Duplicate Records Check")
print("="*50)
duplicates = df.duplicated().sum()
print(f"Duplicate rows: {duplicates}")

if duplicates == 0:
    print("✅ No duplicates found!")

🔄 Duplicate Records Check
Duplicate rows: 0
✅ No duplicates found!


---
## Step 5: Clean and Transform Data

In [11]:
# Convert date column
df['date'] = pd.to_datetime(df['date'])

# Extract time components
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['quarter'] = df['date'].dt.quarter

print("✅ Date column converted!")
print(f"📅 Date range: {df['date'].min().strftime('%B %Y')} to {df['date'].max().strftime('%B %Y')}")

✅ Date column converted!
📅 Date range: January 2007 to October 2023


In [12]:
# Standardise column names (lowercase)
df.columns = df.columns.str.lower().str.replace(' ', '_')

print("✅ Column names standardised:")
print(list(df.columns))

✅ Column names standardised:
['open', 'high', 'low', 'close', 'inflation', 'country', 'iso3', 'date', 'year', 'month', 'quarter']


---
## Step 6: Create New Features (Feature Engineering)

### Why Feature Engineering?
Feature engineering transforms raw data into features that better represent the underlying patterns. This is often **the most impactful step** for improving model performance.

### Features We Create:

| Feature | Formula | Purpose |
|---------|---------|---------|
| `price_range` | High - Low | Measures **volatility** within the period |
| `price_change` | Close - Open | Shows **direction** of price movement |
| `price_change_pct` | (Close - Open) / Open × 100 | Normalises change for comparison across countries |

### Why These Features?
- **Volatility indicators** help identify unstable markets
- **Percentage changes** allow fair comparison between countries with different price levels
- **Temporal features** (year, month, quarter) enable seasonal analysis

In [13]:
# Price volatility (high - low)
df['price_range'] = df['high'] - df['low']

# Price change (close - open)
df['price_change'] = df['close'] - df['open']

# Percentage change
df['price_change_pct'] = ((df['close'] - df['open']) / df['open'] * 100).round(4)

print("✅ New features created:")
print("   • price_range: Volatility indicator (high - low)")
print("   • price_change: Direction of change (close - open)")
print("   • price_change_pct: Percentage change")

✅ New features created:
   • price_range: Volatility indicator (high - low)
   • price_change: Direction of change (close - open)
   • price_change_pct: Percentage change


---
## Step 7: Data Validation

In [14]:
print("🔍 Data Validation")
print("="*50)

# Check for negative prices
price_cols = ['open', 'high', 'low', 'close']
negative_prices = (df[price_cols] < 0).sum().sum()
print(f"Negative prices: {negative_prices}")

# Check high >= low
invalid_range = (df['high'] < df['low']).sum()
print(f"Invalid range (high < low): {invalid_range}")

if negative_prices == 0 and invalid_range == 0:
    print("\n✅ All validation checks passed!")

🔍 Data Validation
Negative prices: 0
Invalid range (high < low): 0

✅ All validation checks passed!


---
## Step 8: Final Summary

In [15]:
print("📊 CLEANED DATA SUMMARY")
print("="*60)
print(f"Total records: {len(df):,}")
print(f"Total columns: {len(df.columns)}")
print(f"Countries: {df['country'].nunique()}")
print(f"Date range: {df['date'].min().strftime('%Y-%m')} to {df['date'].max().strftime('%Y-%m')}")
print(f"\nColumns: {list(df.columns)}")

df.head()

📊 CLEANED DATA SUMMARY
Total records: 4,798
Total columns: 14
Countries: 25
Date range: 2007-01 to 2023-10

Columns: ['open', 'high', 'low', 'close', 'inflation', 'country', 'iso3', 'date', 'year', 'month', 'quarter', 'price_range', 'price_change', 'price_change_pct']


,open,high,low,close,inflation,country,iso3,date,year,month,quarter,price_range,price_change,price_change_pct
0,0.53,0.54,0.53,0.53,NaN,Afghanistan,AFG,2007-01-01,2007,1,1,0.01,0.00,0.0000
1,0.53,0.54,0.53,0.53,NaN,Afghanistan,AFG,2007-02-01,2007,2,1,0.01,0.00,0.0000
2,0.54,0.54,0.53,0.53,NaN,Afghanistan,AFG,2007-03-01,2007,3,1,0.01,-0.01,-1.8519
3,0.53,0.55,0.53,0.55,NaN,Afghanistan,AFG,2007-04-01,2007,4,2,0.02,0.02,3.7736
4,0.56,0.57,0.56,0.57,NaN,Afghanistan,AFG,2007-05-01,2007,5,2,0.01,0.01,1.7857


---
## Step 9: Save Cleaned Data

In [16]:
os.makedirs('../data/cleaned', exist_ok=True)
df.to_csv('../data/cleaned/food_prices_cleaned.csv', index=False)

print("✅ Cleaned data saved!")
print("📁 Location: data/cleaned/food_prices_cleaned.csv")

✅ Cleaned data saved!
📁 Location: data/cleaned/food_prices_cleaned.csv


---
## Summary

| Step | Action | Result |
|------|--------|--------|
| 1 | Load raw data | 4,798 records loaded |
| 2 | Explore structure | 8 original columns |
| 3 | Check missing values | ~7.6% inflation values missing |
| 4 | Check duplicates | No duplicates found |
| 5 | Convert dates | Extracted year, month, quarter |
| 6 | Create features | Added price_range, price_change, price_change_pct |
| 7 | Validate data | All checks passed |
| 8 | Save cleaned data | Ready for analysis |

**Next step:** Open `Data_Analysis.ipynb` to explore the data!